In [2]:
import requests
import pandas as pd
import os

LATITUDE = -26.2041
LONGITUDE = 28.0473


START_DATE = "2022-04-01"
END_DATE = "2026-11-22" 

url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": LATITUDE,
    "longitude": LONGITUDE,
    "start_date": START_DATE,
    "end_date": END_DATE,
    "hourly": ["temperature_2m", "relative_humidity_2m", "wind_speed_10m"],
    "timezone": "Africa/Johannesburg"
}

print(f"Fetching weather data from {START_DATE} to {END_DATE}...")
response = requests.get(url, params=params)

if response.status_code == 200:
    data = response.json()
    weather_df = pd.DataFrame(data['hourly'])
    weather_df.rename(columns={'time': 'timestamp'}, inplace=True)
    weather_df['timestamp'] = pd.to_datetime(weather_df['timestamp'])
    
    weather_df.to_csv('data/joburg_weather.csv', index=False)
    print(f"Success! Saved {len(weather_df)} hours of weather data.")
else:
    print(f"Failed to fetch data. Status code: {response.status_code}")

Fetching weather data from 2022-04-01 to 2026-11-22...


C:\Users\tshia\AppData\Local\Temp\ipykernel_100060\2464343433.py:28: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  weather_df.rename(columns={'time': 'timestamp'}, inplace=True)


Success! Saved 40728 hours of weather data.


In [3]:
eskom_df = pd.read_csv("data/eskom_data.csv")

#Find the date column 
date_col = eskom_df.columns[0]
eskom_df[date_col] = pd.to_datetime(eskom_df[date_col])

#Filter the Eskom dataset to match the weather data timeframe
# April 2022 to 22 November 2026
eskom_df = eskom_df[(eskom_df[date_col] >= '2022-04-01') & (eskom_df[date_col] <= '2026-11-22')]

#Print the column names 
print ("---COLUMN NAMES ---")
for col in eskom_df.columns:
    print(col)

#Show the first 5 rows 
print("\n --n FIRST 5 ROWS ---")
display(eskom_df.head())

---COLUMN NAMES ---
Date Time Hour Beginning
Residual Demand
RSA Contracted Demand
International Imports
Thermal Generation
Nuclear Generation
Eskom OCGT Generation
Pumped Water Generation
Pumped Water SCO Pumping
Wind
PV
CSP
Total RE
Installed Eskom Capacity
Total PCLF
Total UCLF

 --n FIRST 5 ROWS ---


,Date Time Hour Beginning,Residual Demand,RSA Contracted Demand,International Imports,Thermal Generation,Nuclear Generation,Eskom OCGT Generation,Pumped Water Generation,Pumped Water SCO Pumping,Wind,PV,CSP,Total RE,Installed Eskom Capacity,Total PCLF,Total UCLF
0,2022-04-01 00:00:00,21105.801,22240.789,1068.0,20877.0,910.0,0.0,0.0,-2345.0,1091.744,0.0,0.0,1134.988,46797.0,6278.0,12114.285
1,2022-04-01 01:00:00,20936.935,22043.880,1121.0,20709.0,910.0,0.0,0.0,-2399.0,1063.786,0.0,0.0,1106.945,46797.0,6650.0,12540.067
2,2022-04-01 02:00:00,20892.068,21880.597,1295.0,20498.0,910.0,0.0,0.0,-2407.0,956.343,0.0,0.0,988.529,46797.0,6650.0,12216.002
3,2022-04-01 03:00:00,20964.935,21993.023,1337.0,20529.0,910.0,0.0,0.0,-2407.0,986.865,0.0,0.0,1028.088,46797.0,6650.0,12014.730
4,2022-04-01 04:00:00,21886.935,22914.172,1164.0,21312.0,910.0,0.0,0.0,-2095.0,984.148,0.0,0.0,1027.237,46797.0,6650.0,12084.470


In [4]:
# Ensure the Eskom date column is a proper datetime object
eskom_df['Date Time Hour Beginning'] = pd.to_datetime(eskom_df['Date Time Hour Beginning'])

# Rename the Eskom date column to 'timestamp' so it matches the weather data
eskom_df.rename(columns={'Date Time Hour Beginning': 'timestamp'}, inplace=True)

# Merge the two dataframes on the 'timestamp' column
# We use an 'inner' join so we only keep hours where we have BOTH weather and Eskom data
merged_df = pd.merge(weather_df, eskom_df, on='timestamp', how='inner')

print(f"Original weather rows: {len(weather_df)}")
print(f"Original Eskom rows: {len(eskom_df)}")
print(f"Merged rows: {len(merged_df)}")

# Display the first 5 rows of our new merged dataset
display(merged_df.head())

Original weather rows: 40728
Original Eskom rows: 40705
Merged rows: 40705


C:\Users\tshia\AppData\Local\Temp\ipykernel_100060\3255844609.py:5: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  eskom_df.rename(columns={'Date Time Hour Beginning': 'timestamp'}, inplace=True)


,timestamp,temperature_2m,relative_humidity_2m,wind_speed_10m,Residual Demand,RSA Contracted Demand,International Imports,Thermal Generation,Nuclear Generation,Eskom OCGT Generation,Pumped Water Generation,Pumped Water SCO Pumping,Wind,PV,CSP,Total RE,Installed Eskom Capacity,Total PCLF,Total UCLF
0,2022-04-01 00:00:00,14.2,54,7.6,21105.801,22240.789,1068.0,20877.0,910.0,0.0,0.0,-2345.0,1091.744,0.0,0.0,1134.988,46797.0,6278.0,12114.285
1,2022-04-01 01:00:00,13.6,59,8.0,20936.935,22043.880,1121.0,20709.0,910.0,0.0,0.0,-2399.0,1063.786,0.0,0.0,1106.945,46797.0,6650.0,12540.067
2,2022-04-01 02:00:00,12.2,68,10.1,20892.068,21880.597,1295.0,20498.0,910.0,0.0,0.0,-2407.0,956.343,0.0,0.0,988.529,46797.0,6650.0,12216.002
3,2022-04-01 03:00:00,13.3,61,11.6,20964.935,21993.023,1337.0,20529.0,910.0,0.0,0.0,-2407.0,986.865,0.0,0.0,1028.088,46797.0,6650.0,12014.730
4,2022-04-01 04:00:00,12.6,72,13.8,21886.935,22914.172,1164.0,21312.0,910.0,0.0,0.0,-2095.0,984.148,0.0,0.0,1027.237,46797.0,6650.0,12084.470


In [5]:
#Data preprocessing 
missing_values = merged_df.isnull().sum()
#Show columns that only have missing values
missing_values = missing_values[missing_values>0]

if len(missing_values) ==0:
    print("No missing values found! The data is clean.")
else :
    print("Columns with missing values:")
    print(missing_values)

Columns with missing values:
Residual Demand             25
RSA Contracted Demand       25
International Imports       25
Thermal Generation          25
Nuclear Generation          25
Eskom OCGT Generation       25
Pumped Water Generation     25
Pumped Water SCO Pumping    25
Wind                        25
PV                          25
CSP                         25
Total RE                    25
Installed Eskom Capacity    25
Total PCLF                  25
Total UCLF                  25
dtype: int64


In [6]:
#Drop the 25 rows of missing values in the Eskom data
merged_df.dropna(inplace = True)

#Verify if it clean 
print(f"Missing values remaining: {merged_df.isnull().sum().sum()}")
print(f"Final row count: {len(merged_df)}")

#Save the cleaned dataset 
merged_df.to_csv('data/cleaned_merged_data.csv', index=False)
print("Cleaned dataset saved to data/cleaned_merged_data.csv")

C:\Users\tshia\AppData\Local\Temp\ipykernel_100060\1211280833.py:2: Pandas4Warning: The inplace keyword in DataFrame.dropna is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  merged_df.dropna(inplace = True)


Missing values remaining: 0
Final row count: 40680
Cleaned dataset saved to data/cleaned_merged_data.csv


In [11]:
# Feature engineering
# Sort data chronoloically for time series
merged_df = merged_df.sort_values("timestamp").reset_index(drop = True)

#Time Features ( daily vs weekly seasonality)
merged_df["hour"] = merged_df["timestamp"].dt.hour
merged_df["day_of_week"] = merged_df["timestamp"].dt.dayofweek
merged_df["is_weekend"] = merged_df["day_of_week"].apply(lambda x: 1 if x>=5 else 0)
merged_df["month"] = merged_df["timestamp"].dt.month

# Lag Features (What was the demand 1 hour ago? What was it 24 hours ago?)
merged_df ["demand_lag_1"] = merged_df["Residual Demand"].shift(1)
merged_df ["demand_lag_24"] = merged_df["Residual Demand"].shift(24)

#Rolling Features( avg trend over the last 24 hours)
merged_df["temp_rolling_24"] = merged_df["temperature_2m"].rolling(window=24).mean()
merged_df["uclf_rolling_24"] = merged_df["Total UCLF"].rolling(window=24).mean()

#Drop the rows that now have NaNs (because the first 24 hours won't have lag/rolling data)
merged_df.dropna(inplace=True)

print(f"Feature Engineering complete! New dataset shape: { merged_df.shape}")
print("New features created: hour, day_of_week, is_weekend, month, demand_lag_1, demand_lag_24, temp_rolling_24, uclf_rolling_24")

#Save the final feature-engineering dataset
merged_df.to_csv("data/final_features.csv", index = False)
display(merged_df.head())

C:\Users\tshia\AppData\Local\Temp\ipykernel_100060\3557904111.py:7: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  merged_df["day_of_week"] = merged_df["timestamp"].dt.dayofweek
C:\Users\tshia\AppData\Local\Temp\ipykernel_100060\3557904111.py:20: Pandas4Warning: The inplace keyword in DataFrame.dropna is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  merged_df.dropna(inplace=True)


Feature Engineering complete! New dataset shape: (40656, 27)
New features created: hour, day_of_week, is_weekend, month, demand_lag_1, demand_lag_24, temp_rolling_24, uclf_rolling_24


,timestamp,temperature_2m,relative_humidity_2m,wind_speed_10m,Residual Demand,RSA Contracted Demand,International Imports,Thermal Generation,Nuclear Generation,Eskom OCGT Generation,...,Total PCLF,Total UCLF,hour,day_of_week,is_weekend,month,demand_lag_1,demand_lag_24,temp_rolling_24,uclf_rolling_24
24,2022-04-02 00:00:00,13.3,75,10.5,21108.935,22263.697,1271.0,20737.00,912.0,0.0,...,6650.0,11239.027,0,5,1,4,21943.801,21105.801,16.637500,11740.046667
25,2022-04-02 01:00:00,13.4,73,9.7,20692.801,21903.340,1281.0,20308.00,912.0,0.0,...,6735.5,11218.293,1,5,1,4,21108.935,20936.935,16.629167,11684.972750
26,2022-04-02 02:00:00,18.1,54,6.8,20603.935,21803.022,1282.0,20186.00,912.0,0.0,...,6840.0,11048.225,2,5,1,4,20692.801,20892.068,16.875000,11636.315375
27,2022-04-02 03:00:00,15.1,65,8.0,20661.801,21831.765,1296.0,20249.00,912.0,0.0,...,6840.0,10782.225,3,5,1,4,20603.935,20964.935,16.950000,11584.961000
28,2022-04-02 04:00:00,15.1,64,7.6,20984.575,22088.149,1301.0,20529.64,912.0,0.0,...,6840.0,11133.371,4,5,1,4,20661.801,21886.935,17.054167,11545.331875
